<a href="https://colab.research.google.com/github/vguetler/darkweb_topic_modeling/blob/main/darkwebforumstopic2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Topic modeling dark web forums January 2025**

Forums:
*   Ansar
*   Gawaher
*   Islamic network
*   Mywic



Topic models hacker dark web forums using BERTopic To analyze: topics, documents, similarity, topics over time To visualize: topics, clusters, similarity, topics over time -dynamic topic modeling

The BERTopic algorithm contains 3 stages:

1.Embed the textual data(documents) In this step, the algorithm extracts document embeddings with BERT, or it can use any other embedding technique.

By default, it uses the following sentence transformers

"paraphrase-MiniLM-L6-v2"- This is an English BERT-based model trained specifically for semantic similarity tasks. "paraphrase-multilingual-MiniLM-L12-v2"- This is similar to the first, with one major difference is that the xlm models work for 50+ languages.

2.Cluster Documents It uses UMAP to reduce the dimensionality of embeddings and the HDBSCAN technique to cluster reduced embeddings and create clusters of semantically similar documents.

3.Create a topic representation The last step is to extract and reduce topics with class-based TF-IDF and then improve the coherence of words with Maximal Marginal Relevance.



**Enabling the GPU**

First, you'll need to enable GPUs for the notebook:

Navigate to Edit→Notebook Settings select GPU from the Hardware Accelerator drop-down https://colab.research.google.com/notebooks/gpu.ipynb https://colab.research.google.com/drive/1FieRA9fLdkQEGDIMYl0I3MCjSUKVF8C-?usp=sharing#scrollTo=130PIKarkY1_

In [ ]:
# Install bertopic and restart runtime

%%capture
!pip install datamapplot
!pip install bertopic


In [ ]:
# load data from desktop to google collab

# ansar from wvu>diss script folder

from google.colab import files
files.upload()

In [ ]:
# Load data from darkwebpycon or wvu diss script desktop folder


# read the files
import pandas as pd

ansar = pd.read_csv("ansar.csv")

In [ ]:
# view the data

ansar

In [ ]:
# Back to cleaning data
# remove missing

# remove na's
ansar = ansar[ansar['Message'].notnull()]

In [ ]:
ansar.info()

C**reate Models**

To create a model using BERTopic, you need to load the data as a list and then pass it to the fit_transform method. This method will do the following:

Fit the model on the collection of tweets. Generate topics. Return the forums with the topics.

Make sure that the minimum topic size of our topics is 50. We do this to limit the number of topics that could be generated. For example, if the minimum were to be 10 then much more topics could be generated that are most likely to be of little interest. Since we want large topics, we set it to 50. default is 10

BERTopic is stochastic since it uses UMAP as one of its dependencies so the results may differ between runs.

In [ ]:
from bertopic import BERTopic

#convert to list
docs = ansar.Message.to_list()

# create model
topic_model = BERTopic(language="english", calculate_probabilities=True, verbose=True, min_topic_size=40) #min topic size is 50
#topic_model = BERTopic(verbose=True, embedding_model="paraphrase-MiniLM-L12-v2", min_topic_size=50)

topics, probs = topic_model.fit_transform(docs)

In [ ]:
topic_model.get_topic_info()

In [ ]:
# view no of topics = 78 topics after setting min limit to 40, before when 10 over 200 topics
freq = topic_model.get_topic_info()
print("Number of topics: {}".format( len(freq)))
freq.head() # add number to see i.e 10

In [ ]:
#save the results

results = topic_model.get_topic_info()
results.to_csv("topic_model_metadata.csv", index=False) # saved on collab, download. Looks good
display(results)

In [ ]:
topic_model.save("ansar_topic_model")

In [ ]:
# Get overall topic information
#topic_info = topic_model.get_topic_info()

# View a specific topic's keywords
#topic_keywords = topic_model.get_topic(0)  # Replace 0 with the desired topic number

In [ ]:
# @title Topic vs Count

from matplotlib import pyplot as plt
freq.plot(kind='scatter', x='Topic', y='Count', s=32, alpha=.8)
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
topic_model.get_topic_freq().head(11)

In [ ]:
# select a specific topic and get the top n words for that topic and their c-TF-IDF scores.

topic_model.get_topic(10) #stop words!

In [ ]:
# view all the topics

topic_model.get_topic_info()

In [ ]:
# document info

document_info = topic_model.get_document_info(docs)

In [ ]:
document_info.head()

In [ ]:

# Extract the best representing documents per topic

df = pd.DataFrame({"Document": docs, "Topic": topic_model.topics_})

In [ ]:
df.head()

In [ ]:
topic_model.visualize_hierarchy(top_n_topics=50, title= "Ansar Dark Web Forum Hierarchical Clustering") # 50 topics

In [ ]:
# visualize top 10 topics


topic_model.visualize_barchart(top_n_topics=10, title= "Ansar1 Dark Web Forum Topic Word Scores")

In [ ]:
# Custom labels - select topics and label - CHANGE LABELS!!!1

topic_model.set_topic_labels({0: "Hidden content", 1: "Download", 2: "Learn Python", 3: "Cracking", 4: "Software",
                              5: "Syntax", 6: "Languages", 7: "Members", 8: "Crypter/Malware", 9: "Windows Process",
                              10: "Remote Access Trojan", 11: "Cloudfare"})
topic_model.get_topic_info().head(12)

In [ ]:
topic_model.visualize_heatmap(n_clusters=20, width=1000, height=1000)

In [ ]:
# find similar topics - e.g ransomware, ddos, crypter, keylogger, malware, password, carding

similar_topics, similarity = topic_model.find_topics("taliban", top_n=8); similar_topics

In [ ]:
topic_model.get_topic(157)  # view the most similar topics

In [ ]:
# visualize only these

topic_model.visualize_barchart(topics = [20, 100, 149, 112, 259, 257, 3], n_words = 8, title = "Topics Related to Term Taliban: Ansar1 Forum")

# **Visualize documents**

Visualize the documents inside the topics to see if they were assigned correctly or whether they make sense. To do so, we can use the topic_model.visualize_documents() function. This function recalculates the document embeddings and reduces them to 2-dimensional space for easier visualization purposes

In [ ]:
from sentence_transformers import SentenceTransformer
#from bertopic import BERTopic
from umap import UMAP

# Prepare embeddings
#docs = fetch_20newsgroups(subset='all',  remove=('headers', 'footers', 'quotes'))['data']
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = sentence_model.encode(docs, show_progress_bar=False)

# Train BERTopic
topic_model = BERTopic().fit(docs, embeddings)

# Run the visualization with the original embeddings
topic_model.visualize_documents(docs, embeddings=embeddings)

# Reduce dimensionality of embeddings, this step is optional but much faster to perform iteratively:
reduced_embeddings = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine').fit_transform(embeddings)
topic_model.visualize_documents(docs, reduced_embeddings=reduced_embeddings)

# Model 2
- Remove stop words
- decrease topics but without updating models
- label topics

In [ ]:
# start again by removing stopwords

#from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

#ignore English stopwords and infrequent words, increasing the n-gram range to consider topic representations that are made up of one or two words


vectorizer_model = CountVectorizer(stop_words="english", min_df=2)
#vectorizer_model = CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2))
#vectorizer_model = CountVectorizer(stop_words="english")
topic_model2 = BERTopic(vectorizer_model=vectorizer_model, language="english", calculate_probabilities=True, verbose=True, min_topic_size=40)
topics, probs = topic_model2.fit_transform(docs)

In [ ]:
topic_model2.get_topic_info()

In [ ]:
# view no of topics = now 78 topics, with min topic size of 40, still need to clean the text, 8k as outliers, most common words discussed

freq2 = topic_model2.get_topic_info()
print("Number of topics: {}".format( len(freq2)))
freq2.head()

In [ ]:
#save the results

results = topic_model2.get_topic_info()
results.to_csv("topic_model_results2.csv", index=False) # saved on collab, download. Looks good
display(results)

In [ ]:
topic_model2.visualize_barchart(top_n_topics=20)

In [ ]:
# select a specific topic and get the top n words for that topic and their c-TF-IDF scores.

topic_model2.get_topic(-1) #as outlier but seem interesting

In [ ]:
# document info

document_info2 = topic_model2.get_document_info(docs)

In [ ]:
document_info2.head()

# **Visualizations**

BerTopic allows you to visualize the topics that were generated in a way very similar to LDAvis. This will allow you to get more insights into the topic's quality. In this article, we will look at three methods to visualize the topics.

In [ ]:
topic_model2.visualize_hierarchy(top_n_topics=30, title= "Ansar1 Dark Web Forum Hierarchical Clustering")

**Visualize Terms**

We can visualize the selected terms for a few topics by creating bar charts out of the c-TF-IDF scores for each topic representation. Insights can be gained from the relative c-TF-IDF scores between and within topics. Moreover, you can easily compare topic representations to each other

In [ ]:
topic_model2.visualize_barchart(top_n_topics= 12, title= "Ansar1 Dark Web Forum Topic Word Scores")

# **Topic Reduction**

We can also reduce the number of topics after having trained a BERTopic model. The advantage of doing so, is that you can decide the number of topics after knowing how many are actually created. It is difficult to predict before training your model how many topics that are in your documents and how many will be extracted. Instead, we can decide afterwards how many topics seems realistic:

### **Report this in paper**


In [ ]:
# Reduce model2 from 280 to 120

topic_model2.reduce_topics(docs, nr_topics=100)

In [ ]:
# Access the newly updated topics with:
print(topic_model2.topics_)

In [ ]:
# view no of topics = now 90 topics, still need to clean the text,  9k as outliers, most common words discussed

freq3 = topic_model2.get_topic_info()
print("Number of topics: {}".format( len(freq3)))
freq3.head()

In [ ]:
topic_model2.visualize_hierarchy(top_n_topics=50, title= "Ansar1 Dark Web Forum Hierarchical Clustering")

In [ ]:
topic_model2.visualize_barchart(top_n_topics= 12, n_words = 8, title= "Ansar1 Dark Web Forum Topic Word Scores")

In [ ]:
# Custom labels - select 10 topics and label CHECK THIS NEW MODELS NEED NEW LABELS

topic_model2.set_topic_labels({0: "Somalia", 1: "Baghdad", 2: "Hamas", 3: "Downloads", 4: "Russia",
                              5: "Amir", 6: "Militants", 7: "Translations", 8: "Pakistan", 9: "Video"})
                              #10: "Pakistan", 11: "Logo"})
topic_model2.get_topic_info().head(12) #save this as csv file for further analysis?


In [ ]:
topic_model2.visualize_barchart(top_n_topics= 10, n_words = 8, custom_labels=True, title= "Ansar1 Dark Web Forum Topics: Word Scores")

In [ ]:
# find similar topics - e.g taliban, hamas, jihad, mujahideen

similar_topics, similarity = topic_model2.find_topics("taliban", top_n=8); similar_topics

In [ ]:
topic_model2.get_topic(6)  # view the most similar topics

In [ ]:
# visualize only these

topic_model2.visualize_barchart(topics = [58, 8, -1, 25, 18, 72, 13, 62], n_words = 8, title = "Topics Related to Term Taliban: Ansar1 Forum")

In [ ]:
# find similar topics - e.g taliban, hamas, jihad, mujahideen

similar_topics, similarity = topic_model2.find_topics("mujahideen", top_n=8); similar_topics

In [ ]:
topic_model2.get_topic(16)  # view the most similar topics

In [ ]:
# visualize only these

topic_model2.visualize_barchart(topics = [65, 14, 22, -1, 38, 46, 64], n_words = 8, title = "Topics Related to Term Mujahideen: Ansar1 Forum")

In [ ]:
# find similar topics - e.g taliban, hamas, jihad, mujahideen

similar_topics, similarity = topic_model2.find_topics("terrorists", top_n=8); similar_topics

In [ ]:
topic_model2.get_topic(7)  # view the most similar topics

In [ ]:
# visualize only these

topic_model2.visualize_barchart(topics = [6, 33, -1, 25, 13, 21, 66], n_words = 8, title = "Topics Related to Term Terrorism: Ansar1 Forum")

In [ ]:
# find similar topics - e.g taliban, hamas, jihad, mujahideen

similar_topics, similarity = topic_model2.find_topics("jihad", top_n=8); similar_topics

In [ ]:
topic_model2.get_topic(71)  # view the most similar topics

In [ ]:
# visualize only these

topic_model2.visualize_barchart(topics = [64, 48, 14, 76, -1, 41, 46, 16], n_words = 8, title = "Topics Related to Term Jihad: Ansar1 Forum")

**Visualize documents**

Visualize the documents inside the topics to see if they were assigned correctly or whether they make sense. To do so, we can use the topic_model.visualize_documents() function. This function recalculates the document embeddings and reduces them to 2-dimensional space for easier visualization purposes

In [ ]:
from sentence_transformers import SentenceTransformer
#from bertopic import BERTopic
from umap import UMAP

# Prepare embeddings
#docs = fetch_20newsgroups(subset='all',  remove=('headers', 'footers', 'quotes'))['data']
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = sentence_model.encode(docs, show_progress_bar=False)

# Train BERTopic
topic_model = BERTopic().fit(docs, embeddings)

# Run the visualization with the original embeddings
topic_model2.visualize_documents(docs, embeddings=embeddings)

# Reduce dimensionality of embeddings, this step is optional but much faster to perform iteratively:
reduced_embeddings = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine').fit_transform(embeddings)
topic_model2.visualize_documents(docs, reduced_embeddings=reduced_embeddings)

In [ ]:
# check if same shape, rows, columns
embeddings.shape, reduced_embeddings.shape

In [ ]:
import matplotlib
#matplotlib.rcParams["figure.dpi"] = 72

In [ ]:
# pip install datamapplot

In [ ]:
import datamapplot

## https://datamapplot.readthedocs.io/en/latest/demo.html

In [ ]:
# pip install bertopic

In [ ]:
# Visualize documents with DataMapPlot
# with the original embeddings
topic_model2.visualize_document_datamap(docs, embeddings=embeddings,
    sub_title="A data map of posts from the Ansar1 dark web forum")

In [ ]:
# with the original embeddings
#topic_model.visualize_document_datamap(docs, embeddings=embeddings)

# with the reduced embeddings
topic_model2.visualize_document_datamap(docs, reduced_embeddings=reduced_embeddings)

In [ ]:
#fig = topic_model.visualize_document_datamap(docs, reduced_embeddings=reduced_embeddings) # taking forever!!!
#fig.savefig("Desktop/file.png", bbox_inches="tight")